In [0]:
-- create or replace table com_edp_prd.cmpa_insights_internal_schema.rpt_dashboard_trend_data as

-- with patient_events_base as (

--     select
--         'Account_Leads' as dashboard_name,
--         'Time_Series' as section_name,
--         'PATIENT_COUNT' as metric_type,
--         'ELAPRASE' as product_name,
--         primary_hcp_territory_id_2yr as territory_id,
--         primary_hcp_territory_2yr as territory_name,
--         primary_hcp_region_id_2yr as region_id,
--         primary_hcp_region_2yr as region_name,
--         'MONTHLY' as time_period,
--         cast(date_trunc('month', first_incidence_treatment_date) as date) as period_start_dt,
--         count(distinct patient_id) as metric_value
--     from com_edp_prd.cmpa_insights_internal_schema.patient360_master
--     where first_incidence_treatment_date is not null
--       and patient_age < 17
--       and primary_hcp_territory_id_2yr is not null
--     group by
--         primary_hcp_territory_id_2yr,
--         primary_hcp_territory_2yr,
--         primary_hcp_region_id_2yr,
--         primary_hcp_region_2yr,
--         cast(date_trunc('month', first_incidence_treatment_date) as date)

--     union all

--     select
--         'Account_Leads' as dashboard_name,
--         'Time_Series' as section_name,
--         'PATIENT_COUNT' as metric_type,
--         'AVLAYAH' as product_name,
--         primary_hcp_territory_id_2yr as territory_id,
--         primary_hcp_territory_2yr as territory_name,
--         primary_hcp_region_id_2yr as region_id,
--         primary_hcp_region_2yr as region_name,
--         'MONTHLY' as time_period,
--         cast(date_trunc('month', tivi_first_incidence_treatment_date) as date) as period_start_dt,
--         count(distinct patient_id) as metric_value
--     from com_edp_prd.cmpa_insights_internal_schema.patient360_master
--     where tivi_first_incidence_treatment_date is not null
--       and patient_age < 17
--       and primary_hcp_territory_id_2yr is not null
--     group by
--         primary_hcp_territory_id_2yr,
--         primary_hcp_territory_2yr,
--         primary_hcp_region_id_2yr,
--         primary_hcp_region_2yr,
--         cast(date_trunc('month', tivi_first_incidence_treatment_date) as date)
-- ),

-- patient_events as (
--     select
--         dashboard_name,
--         section_name,
--         metric_type,
--         product_name,
--         territory_id,
--         territory_name,
--         region_id,
--         region_name,
--         time_period,
--         period_start_dt,
--         date_format(period_start_dt, 'MMM-yyyy') as period_month,
--         metric_value
--     from patient_events_base
-- ),

-- base_dispense as (
--     select
--         d.*,
--         z.region_id,
--         z.region_name,
--         z.territory_id,
--         z.territory_name
--     from com_edp_prd.cmpa_insights_internal_schema.qa_sp_dispense_dummy d
--     left join com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
--         on d.shipment_hcp_zip = z.zipcode
--     where z.territory_id is not null
-- ),

-- weekly_vials_base as (
--     select
--         'Account_Leads' as dashboard_name,
--         'Time_Series' as section_name,
--         'VIAL_COUNT' as metric_type,
--         'AVLAYAH' as product_name,
--         territory_id,
--         territory_name,
--         region_id,
--         region_name,
--         'WEEKLY' as time_period,
--         cast(date_sub(ship_date, pmod(dayofweek(ship_date) - 6, 7)) as date) as period_start_dt,
--         sum(quantity) as metric_value
--     from base_dispense
--     where ship_date is not null
--     group by
--         territory_id,
--         territory_name,
--         region_id,
--         region_name,
--         cast(date_sub(ship_date, pmod(dayofweek(ship_date) - 6, 7)) as date)
-- ),

-- weekly_vials as (
--     select
--         dashboard_name,
--         section_name,
--         metric_type,
--         product_name,
--         territory_id,
--         territory_name,
--         region_id,
--         region_name,
--         time_period,
--         period_start_dt,
--         date_format(period_start_dt, 'MMM-yyyy') as period_month,
--         metric_value
--     from weekly_vials_base
-- )

-- select
--     dashboard_name,
--     section_name,
--     metric_type,
--     product_name,
--     territory_id,
--     territory_name,
--     region_id,
--     region_name,
--     time_period,
--     period_start_dt,
--     period_month,
--     metric_value
-- from patient_events

-- union all

-- select
--     dashboard_name,
--     section_name,
--     metric_type,
--     product_name,
--     territory_id,
--     territory_name,
--     region_id,
--     region_name,
--     time_period,
--     period_start_dt,
--     period_month,
--     metric_value
-- from weekly_vials;

In [0]:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.rpt_dashboard_trend_data AS

-- WITH date_spine AS (

--     -- MONTHLY
--     SELECT DISTINCT
--         'MONTHLY' AS time_period,
--         CAST(date_trunc('month', d) AS DATE) AS period_start_dt
--     FROM (
--         SELECT explode(sequence(to_date('2023-01-01'), current_date(), interval 1 month)) AS d
--     )

--     UNION ALL

--     -- WEEKLY (currently Friday-based; change if needed)
--     SELECT DISTINCT
--         'WEEKLY' AS time_period,
--         CAST(date_sub(d, pmod(dayofweek(d) - 6, 7)) AS DATE) AS period_start_dt
--     FROM (
--         SELECT explode(sequence(to_date('2023-01-01'), current_date(), interval 1 week)) AS d
--     )
-- ),

-- territory_master AS (
--     SELECT DISTINCT
--         primary_hcp_territory_id_2yr AS territory_id,
--         primary_hcp_territory_2yr AS territory_name,
--         primary_hcp_region_id_2yr AS region_id,
--         primary_hcp_region_2yr AS region_name
--     FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
--     WHERE primary_hcp_territory_id_2yr IS NOT NULL
-- ),

-- /* =========================
--    METRIC DIMENSION (FIX)
--    ========================= */

-- metric_dim AS (
--     SELECT 'PATIENT_COUNT' AS metric_type, 'ELAPRASE' AS product_name
--     UNION ALL
--     SELECT 'PATIENT_COUNT', 'AVLAYAH'
--     UNION ALL
--     SELECT 'VIAL_COUNT', 'AVLAYAH'
-- ),

-- /* =========================
--    PATIENT EVENTS
--    ========================= */

-- patient_events_base AS (

--     -- ELAPRASE
--     SELECT
--         'Account_Leads' AS dashboard_name,
--         'Time_Series' AS section_name,
--         'PATIENT_COUNT' AS metric_type,
--         'ELAPRASE' AS product_name,
--         primary_hcp_territory_id_2yr AS territory_id,
--         primary_hcp_territory_2yr AS territory_name,
--         primary_hcp_region_id_2yr AS region_id,
--         primary_hcp_region_2yr AS region_name,
--         'MONTHLY' AS time_period,
--         CAST(date_trunc('month', first_incidence_treatment_date) AS DATE) AS period_start_dt,
--         COUNT(DISTINCT patient_id) AS metric_value
--     FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
--     WHERE first_incidence_treatment_date IS NOT NULL
--       AND patient_age < 17
--       AND primary_hcp_territory_id_2yr IS NOT NULL
--     GROUP BY
--         primary_hcp_territory_id_2yr,
--         primary_hcp_territory_2yr,
--         primary_hcp_region_id_2yr,
--         primary_hcp_region_2yr,
--         CAST(date_trunc('month', first_incidence_treatment_date) AS DATE)

--     UNION ALL

--     -- AVLAYAH
--     SELECT
--         'Account_Leads',
--         'Time_Series',
--         'PATIENT_COUNT',
--         'AVLAYAH',
--         primary_hcp_territory_id_2yr,
--         primary_hcp_territory_2yr,
--         primary_hcp_region_id_2yr,
--         primary_hcp_region_2yr,
--         'MONTHLY',
--         CAST(date_trunc('month', tivi_first_incidence_treatment_date) AS DATE),
--         COUNT(DISTINCT patient_id)
--     FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
--     WHERE tivi_first_incidence_treatment_date IS NOT NULL
--       AND patient_age < 17
--       AND primary_hcp_territory_id_2yr IS NOT NULL
--     GROUP BY
--         primary_hcp_territory_id_2yr,
--         primary_hcp_territory_2yr,
--         primary_hcp_region_id_2yr,
--         primary_hcp_region_2yr,
--         CAST(date_trunc('month', tivi_first_incidence_treatment_date) AS DATE)
-- ),

-- patient_events AS (
--     SELECT *,
--         date_format(period_start_dt, 'MMM-yyyy') AS period_month
--     FROM patient_events_base
-- ),

-- /* =========================
--    DISPENSE / VIALS
--    ========================= */

-- base_dispense AS (
--     SELECT
--         d.*,
--         z.region_id,
--         z.region_name,
--         z.territory_id,
--         z.territory_name
--     FROM com_edp_prd.cmpa_insights_internal_schema.qa_sp_dispense_dummy d
--     LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
--         ON d.shipment_hcp_zip = z.zipcode
--     WHERE z.territory_id IS NOT NULL
-- ),

-- weekly_vials_base AS (
--     SELECT
--         'Account_Leads' AS dashboard_name,
--         'Time_Series' AS section_name,
--         'VIAL_COUNT' AS metric_type,
--         'AVLAYAH' AS product_name,
--         territory_id,
--         territory_name,
--         region_id,
--         region_name,
--         'WEEKLY' AS time_period,
--         CAST(date_sub(ship_date, pmod(dayofweek(ship_date) - 6, 7)) AS DATE) AS period_start_dt,
--         SUM(quantity) AS metric_value
--     FROM base_dispense
--     WHERE ship_date IS NOT NULL
--     GROUP BY
--         territory_id,
--         territory_name,
--         region_id,
--         region_name,
--         CAST(date_sub(ship_date, pmod(dayofweek(ship_date) - 6, 7)) AS DATE)
-- ),

-- weekly_vials AS (
--     SELECT *,
--         date_format(period_start_dt, 'MMM-yyyy') AS period_month
--     FROM weekly_vials_base
-- ),

-- /* =========================
--    FINAL METRICS
--    ========================= */

-- final_metrics AS (
--     SELECT * FROM patient_events
--     UNION ALL
--     SELECT * FROM weekly_vials
-- ),

-- /* =========================
--    FULL GRID (FIXED)
--    ========================= */

-- full_grid AS (
--     SELECT
--         t.territory_id,
--         t.territory_name,
--         t.region_id,
--         t.region_name,
--         d.time_period,
--         d.period_start_dt,
--         date_format(d.period_start_dt, 'MMM-yyyy') AS period_month,
--         m.metric_type,
--         m.product_name
--     FROM territory_master t
--     CROSS JOIN date_spine d
--     CROSS JOIN metric_dim m
--     WHERE NOT (m.metric_type = 'VIAL_COUNT' AND m.product_name = 'ELAPRASE') -- avoid invalid combo
-- )

-- /* =========================
--    FINAL OUTPUT
--    ========================= */

-- SELECT
--     'Account_Leads' AS dashboard_name,
--     'Time_Series' AS section_name,
--     g.metric_type,
--     g.product_name,
--     g.territory_id,
--     g.territory_name,
--     g.region_id,
--     g.region_name,
--     g.time_period,
--     g.period_start_dt,
--     g.period_month,
--     COALESCE(f.metric_value, 0) AS metric_value

-- FROM full_grid g

-- LEFT JOIN final_metrics f
--     ON g.territory_id = f.territory_id
--     AND g.period_start_dt = f.period_start_dt
--     AND g.time_period = f.time_period
--     AND g.metric_type = f.metric_type
--     AND g.product_name = f.product_name
-- ;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.rpt_dashboard_trend_data AS

WITH date_spine AS (

    -- MONTHLY
    SELECT DISTINCT
        'MONTHLY' AS time_period,
        CAST(date_trunc('month', d) AS DATE) AS period_start_dt
    FROM (
        SELECT explode(sequence(to_date('2023-08-01'), current_date(), interval 1 month)) AS d
    )

    UNION ALL

    -- WEEKLY (currently Friday-based; change if needed)
    SELECT DISTINCT
        'WEEKLY' AS time_period,
        CAST(date_sub(d, pmod(dayofweek(d) - 6, 7)) AS DATE) AS period_start_dt
    FROM (
        SELECT explode(sequence(to_date('2023-08-01'), current_date(), interval 1 week)) AS d
    )
),

territory_master AS (
    SELECT DISTINCT
        primary_hcp_territory_id_2yr AS territory_id,
        primary_hcp_territory_2yr AS territory_name,
        primary_hcp_region_id_2yr AS region_id,
        primary_hcp_region_2yr AS region_name
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
    WHERE primary_hcp_territory_id_2yr IS NOT NULL
),

metric_dim AS (
    SELECT 'PATIENT_COUNT' AS metric_type, 'ELAPRASE' AS product_name
    UNION ALL
    SELECT 'PATIENT_COUNT', 'AVLAYAH'
    UNION ALL
    SELECT 'VIAL_COUNT', 'AVLAYAH'
),

patient_events_base AS (

    -- ELAPRASE
    SELECT
        'Account_Leads' AS dashboard_name,
        'Time_Series' AS section_name,
        'PATIENT_COUNT' AS metric_type,
        'ELAPRASE' AS product_name,
        primary_hcp_territory_id_2yr AS territory_id,
        primary_hcp_territory_2yr AS territory_name,
        primary_hcp_region_id_2yr AS region_id,
        primary_hcp_region_2yr AS region_name,
        'MONTHLY' AS time_period,
        CAST(date_trunc('month', first_incidence_treatment_date) AS DATE) AS period_start_dt,
        COUNT(DISTINCT patient_id) AS metric_value
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
    -- WHERE first_incidence_treatment_date IS NOT NULL
      where patient_age < 17
    --   AND primary_hcp_territory_id_2yr IS NOT NULL
    GROUP BY
        primary_hcp_territory_id_2yr,
        primary_hcp_territory_2yr,
        primary_hcp_region_id_2yr,
        primary_hcp_region_2yr,
        CAST(date_trunc('month', first_incidence_treatment_date) AS DATE)

    UNION ALL

    -- AVLAYAH
    SELECT
        'Account_Leads',
        'Time_Series',
        'PATIENT_COUNT',
        'AVLAYAH',
        primary_hcp_territory_id_2yr,
        primary_hcp_territory_2yr,
        primary_hcp_region_id_2yr,
        primary_hcp_region_2yr,
        'MONTHLY',
        CAST(date_trunc('month', tivi_first_incidence_treatment_date) AS DATE),
        COUNT(DISTINCT patient_id)
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
    -- WHERE tivi_first_incidence_treatment_date IS NOT NULL
      WHERE patient_age < 17
    --   AND primary_hcp_territory_id_2yr IS NOT NULL
    GROUP BY
        primary_hcp_territory_id_2yr,
        primary_hcp_territory_2yr,
        primary_hcp_region_id_2yr,
        primary_hcp_region_2yr,
        CAST(date_trunc('month', tivi_first_incidence_treatment_date) AS DATE)
),

patient_events AS (
    SELECT *,
        date_format(period_start_dt, 'MMM-yyyy') AS period_month
    FROM patient_events_base
),

accounts_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY crx_account_id 
                   ORDER BY ingestion_date DESC
               ) AS rn
        FROM com_edp_prd.com_intgr.distribution_accounts
    )
    WHERE rn = 1
),

shipments_dedup AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY invoice_number
                   ORDER BY ingestion_date DESC
               ) AS rn
        FROM com_edp_prd.com_intgr.distribution_sd_shipments
        WHERE ndc = '84976-0001-01' 
    )
    WHERE rn = 1
),

base_dispense AS (
    SELECT
        CAST(s.invoicedate AS DATE) AS ship_date,
        s.quantity_shipped AS quantity,
        z.region_id,
        z.region_name,
        z.territory_id,
        z.territory_name
    FROM shipments_dedup s
    LEFT JOIN accounts_dedup a
        ON s.crx_account_id = a.crx_account_id
    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON LEFT(REGEXP_REPLACE(a.account_facility_zip, '[^0-9]', ''), 5) =
           LEFT(REGEXP_REPLACE(z.zipcode, '[^0-9]', ''), 5)
    WHERE z.territory_id IS NOT NULL
),

weekly_vials_base AS (
    SELECT
        'Account_Leads' AS dashboard_name,
        'Time_Series' AS section_name,
        'VIAL_COUNT' AS metric_type,
        'AVLAYAH' AS product_name,
        territory_id,
        territory_name,
        region_id,
        region_name,
        'WEEKLY' AS time_period,
        CAST(date_sub(ship_date, pmod(dayofweek(ship_date) - 6, 7)) AS DATE) AS period_start_dt,
        SUM(quantity) AS metric_value
    FROM base_dispense
    WHERE ship_date IS NOT NULL
    GROUP BY
        territory_id,
        territory_name,
        region_id,
        region_name,
        CAST(date_sub(ship_date, pmod(dayofweek(ship_date) - 6, 7)) AS DATE)
),

weekly_vials AS (
    SELECT *,
        date_format(period_start_dt, 'MMM-yyyy') AS period_month
    FROM weekly_vials_base
),

final_metrics AS (
    SELECT * FROM patient_events
    UNION ALL
    SELECT * FROM weekly_vials
),

full_grid AS (
    SELECT
        t.territory_id,
        t.territory_name,
        t.region_id,
        t.region_name,
        d.time_period,
        d.period_start_dt,
        date_format(d.period_start_dt, 'MMM-yyyy') AS period_month,
        m.metric_type,
        m.product_name
    FROM territory_master t
    CROSS JOIN date_spine d
    CROSS JOIN metric_dim m
)

SELECT
    'Account_Leads' AS dashboard_name,
    'Time_Series' AS section_name,
    g.metric_type,
    g.product_name,
    g.territory_id,
    g.territory_name,
    g.region_id,
    g.region_name,
    g.time_period,
    g.period_start_dt,
    g.period_month,
    COALESCE(f.metric_value, 0) AS metric_value

FROM full_grid g

LEFT JOIN final_metrics f
    ON g.territory_id = f.territory_id
    AND g.period_start_dt = f.period_start_dt
    AND g.time_period = f.time_period
    AND g.metric_type = f.metric_type
    AND g.product_name = f.product_name

In [0]:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.rpt_dashboard_trend_data AS
 
-- WITH date_spine AS (
--     SELECT DISTINCT
--         'MONTHLY' AS time_period,
--         CAST(date_trunc('month', d) AS DATE) AS period_start_dt
--     FROM (
--         SELECT explode(sequence(to_date('2023-08-01'), current_date(), interval 1 month)) AS d
--     )
--     UNION ALL
--     SELECT DISTINCT
--         'WEEKLY' AS time_period,
--         CAST(date_sub(d, pmod(dayofweek(d) - 6, 7)) AS DATE) AS period_start_dt
--     FROM (
--         SELECT explode(sequence(to_date('2023-08-01'), current_date(), interval 1 week)) AS d
--     )
-- ),
 
-- territory_master AS (
--     SELECT DISTINCT
--         primary_hcp_territory_id_2yr AS territory_id,
--         primary_hcp_territory_2yr    AS territory_name,
--         primary_hcp_region_id_2yr    AS region_id,
--         primary_hcp_region_2yr       AS region_name
--     FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
--     WHERE primary_hcp_territory_id_2yr IS NOT NULL
-- ),
 
-- metric_dim AS (
--     SELECT 'PATIENT_COUNT' AS metric_type, 'ELAPRASE' AS product_name, 'MONTHLY' AS time_period
--     UNION ALL
--     SELECT 'PATIENT_COUNT', 'AVLAYAH',  'MONTHLY'
--     UNION ALL
--     SELECT 'VIAL_COUNT',    'AVLAYAH',  'WEEKLY'
-- ),
 
-- avlayah_first_ndc_event AS (
--     SELECT
--         patient_id,
--         MIN(event_dt) AS first_avlayah_ndc_dt
--     FROM (
--         SELECT
--             patient_id,
--             CAST(service_date AS DATE) AS event_dt
--         FROM com_edp_prd.com_raw.kom_medical_events
--         WHERE ndc11 = '8497600101'
--           AND service_date >= '2026-03-01'
--         UNION ALL
--         SELECT
--             patient_id,
--             CAST(fill_date AS DATE) AS event_dt
--         FROM com_edp_prd.com_raw.kom_pharmacy_events
--         WHERE ndc11 = '8497600101'
--           AND UPPER(transaction_result) = 'PAID'
--           AND fill_date >= '2026-03-01'
--     )
--     GROUP BY patient_id
-- ),
 
-- avlayah_patients_in_cohort AS (
--     SELECT
--         p.patient_id,
--         p.primary_hcp_territory_id_2yr AS territory_id,
--         p.primary_hcp_territory_2yr    AS territory_name,
--         p.primary_hcp_region_id_2yr    AS region_id,
--         p.primary_hcp_region_2yr       AS region_name,
--         p.patient_age,
--         a.first_avlayah_ndc_dt
--     FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master p
--     INNER JOIN avlayah_first_ndc_event a
--         ON p.patient_id = a.patient_id
-- ),
 
-- patient_events_base AS (
--     SELECT
--         'Account_Leads' AS dashboard_name,
--         'Time_Series'   AS section_name,
--         'PATIENT_COUNT' AS metric_type,
--         'ELAPRASE'      AS product_name,
--         primary_hcp_territory_id_2yr AS territory_id,
--         primary_hcp_territory_2yr    AS territory_name,
--         primary_hcp_region_id_2yr    AS region_id,
--         primary_hcp_region_2yr       AS region_name,
--         'MONTHLY' AS time_period,
--         CAST(date_trunc('month', first_incidence_treatment_date) AS DATE) AS period_start_dt,
--         COUNT(DISTINCT patient_id) AS metric_value
--     FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master
--     WHERE first_incidence_treatment_date IS NOT NULL
--       AND patient_age < 17
--       AND primary_hcp_territory_id_2yr IS NOT NULL
--     GROUP BY
--         primary_hcp_territory_id_2yr,
--         primary_hcp_territory_2yr,
--         primary_hcp_region_id_2yr,
--         primary_hcp_region_2yr,
--         CAST(date_trunc('month', first_incidence_treatment_date) AS DATE)
 
--     UNION ALL
 
--     SELECT
--         'Account_Leads',
--         'Time_Series',
--         'PATIENT_COUNT',
--         'AVLAYAH',
--         territory_id,
--         territory_name,
--         region_id,
--         region_name,
--         'MONTHLY',
--         CAST(date_trunc('month', first_avlayah_ndc_dt) AS DATE),
--         COUNT(DISTINCT patient_id)
--     FROM avlayah_patients_in_cohort
--     WHERE first_avlayah_ndc_dt IS NOT NULL
--       AND patient_age < 17
--       AND territory_id IS NOT NULL
--     GROUP BY
--         territory_id,
--         territory_name,
--         region_id,
--         region_name,
--         CAST(date_trunc('month', first_avlayah_ndc_dt) AS DATE)
-- ),
 
-- patient_events AS (
--     SELECT
--         *,
--         date_format(period_start_dt, 'MMM-yyyy') AS period_month
--     FROM patient_events_base
-- ),
 
-- accounts_dedup AS (
--     SELECT *
--     FROM (
--         SELECT
--             *,
--             ROW_NUMBER() OVER (
--                 PARTITION BY crx_account_id
--                 ORDER BY ingestion_date DESC
--             ) AS rn
--         FROM com_edp_prd.com_intgr.distribution_accounts
--         WHERE is_current = true
--     )
--     WHERE rn = 1
-- ),
 
-- shipments_dedup AS (
--     SELECT *
--     FROM (
--         SELECT
--             *,
--             ROW_NUMBER() OVER (
--                 PARTITION BY invoice_number
--                 ORDER BY ingestion_date DESC
--             ) AS rn
--         FROM com_edp_prd.com_intgr.distribution_sd_shipments
--         WHERE ndc = '84976-0001-01'
--           AND is_current = true
--     )
--     WHERE rn = 1
-- ),
 
-- zip_map AS (
--     SELECT DISTINCT
--         LEFT(REGEXP_REPLACE(zipcode, '[^0-9]', ''), 5) AS zip5,
--         region_id,
--         region_name,
--         territory_id,
--         territory_name
--     FROM com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping
-- ),
 
-- base_dispense AS (
--     SELECT
--         CAST(s.invoicedate AS DATE) AS ship_date,
--         s.quantity_shipped AS quantity,
--         z.region_id,
--         z.region_name,
--         z.territory_id,
--         z.territory_name
--     FROM shipments_dedup s
--     LEFT JOIN accounts_dedup a
--         ON s.crx_account_id = a.crx_account_id
--     LEFT JOIN zip_map z
--         ON LEFT(REGEXP_REPLACE(s.ship_to_address_postal_code, '[^0-9]', ''), 5) = z.zip5
--     WHERE z.territory_id IS NOT NULL
-- ),
 
-- weekly_vials_base AS (
--     SELECT
--         'Account_Leads' AS dashboard_name,
--         'Time_Series'   AS section_name,
--         'VIAL_COUNT'    AS metric_type,
--         'AVLAYAH'       AS product_name,
--         territory_id,
--         territory_name,
--         region_id,
--         region_name,
--         'WEEKLY' AS time_period,
--         CAST(date_sub(ship_date, pmod(dayofweek(ship_date) - 6, 7)) AS DATE) AS period_start_dt,
--         SUM(quantity) AS metric_value
--     FROM base_dispense
--     WHERE ship_date IS NOT NULL
--     GROUP BY
--         territory_id,
--         territory_name,
--         region_id,
--         region_name,
--         CAST(date_sub(ship_date, pmod(dayofweek(ship_date) - 6, 7)) AS DATE)
-- ),
 
-- weekly_vials AS (
--     SELECT
--         *,
--         date_format(period_start_dt, 'MMM-yyyy') AS period_month
--     FROM weekly_vials_base
-- ),
 
-- final_metrics AS (
--     SELECT * FROM patient_events
--     UNION ALL
--     SELECT * FROM weekly_vials
-- ),
 
-- full_grid AS (
--     SELECT
--         t.territory_id,
--         t.territory_name,
--         t.region_id,
--         t.region_name,
--         m.time_period,
--         d.period_start_dt,
--         date_format(d.period_start_dt, 'MMM-yyyy') AS period_month,
--         m.metric_type,
--         m.product_name
--     FROM territory_master t
--     CROSS JOIN metric_dim m
--     INNER JOIN date_spine d
--         ON d.time_period = m.time_period
-- )
 
-- SELECT
--     'Account_Leads' AS dashboard_name,
--     'Time_Series'   AS section_name,
--     g.metric_type,
--     g.product_name,
--     g.territory_id,
--     g.territory_name,
--     g.region_id,
--     g.region_name,
--     g.time_period,
--     g.period_start_dt,
--     g.period_month,
--     COALESCE(f.metric_value, 0) AS metric_value
-- FROM full_grid g
-- LEFT JOIN final_metrics f
--     ON g.territory_id    = f.territory_id
--    AND g.period_start_dt = f.period_start_dt
--    AND g.time_period     = f.time_period
--    AND g.metric_type     = f.metric_type
--    AND g.product_name    = f.product_name;
 
 

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.rpt_dashboard_trend_data

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.rpt_dashboard_trend_data
where metric_type = 'PATIENT_COUNT'
and product_name = 'AVLAYAH'
and metric_value > 0